# 02.2 - Features de historial previo del paciente

En el notebook 02 generamos el label  por visita.
Aqui anadimos features que capturan el **historial previo** de cada paciente antes de cada visita:

-  - cuantas veces habia ido antes a urgencias
-  - dias desde la visita anterior (vacio si es la primera)
-  - si alguna vez fue readmitido antes de esta visita (0/1)

Usamos **todas las visitas** (no solo la primera), lo que da mas datos y mas informacion al modelo.

In [ ]:
import pandas as pd

df = pd.read_csv("edstays_with_target.csv", parse_dates=["intime"])

print(f"Total visitas: {len(df):,}")
print(f"Pacientes unicos: {df['subject_id'].nunique():,}")
df.head()

## Calcular features de historial

Para cada visita, miramos las visitas **anteriores** del mismo paciente.
Ordenamos por paciente y fecha para que el calculo sea correcto.

In [ ]:
# Ordenar por paciente y fecha de visita
df = df.sort_values(["subject_id", "intime"]).reset_index(drop=True)

# Numero de visitas previas (la primera visita tiene 0)
df["n_visitas_previas"] = df.groupby("subject_id").cumcount()

# Dias desde la ultima visita
df["dias_desde_ultima_visita"] = (
    df.groupby("subject_id")["intime"]
    .diff()
    .dt.total_seconds() / 86400
)

# Si alguna visita anterior fue readmision
df["readmitido_antes"] = (
    df.groupby("subject_id")["readmitted"]
    .shift(1)
    .fillna(0)
    .groupby(df["subject_id"])
    .cummax()
    .astype(int)
)

print(df[["subject_id", "intime", "n_visitas_previas",
          "dias_desde_ultima_visita", "readmitido_antes"]].head(15))

## Verificacion

Comprobamos que los calculos tienen sentido: la primera visita de cada paciente debe tener ,  y .

In [ ]:
primeras = df[df["n_visitas_previas"] == 0]
print(f"Primeras visitas: {len(primeras):,}")
print(f"  dias_desde_ultima_visita nulos: {primeras['dias_desde_ultima_visita'].isna().sum():,}  (correcto: todos)")
print(f"  readmitido_antes = 0: {(primeras['readmitido_antes']==0).sum():,}  (correcto: todos)")

print(f"
Distribucion de n_visitas_previas:")
print(df["n_visitas_previas"].describe().round(1))

print(f"
Visitas con readmitido_antes=1: {df['readmitido_antes'].sum():,}  ({df['readmitido_antes'].mean()*100:.1f}%)")

## Guardar

Guardamos el dataset con el historial anadido. El notebook 03 puede usar este archivo en lugar de  para tener estas features disponibles.

In [ ]:
df.to_csv("edstays_with_history.csv", index=False, encoding="utf-8")
print("Guardado: edstays_with_history.csv")
print(f"Shape: {df.shape}")
print(f"Columnas nuevas: n_visitas_previas, dias_desde_ultima_visita, readmitido_antes")